# nb06 — severidade, taxonomia e o campo de visão do instrumento (T30)

**O que este notebook faz:** lê os scores N1/N2 da bateria principal (288 execuções: 18 cenários
de test × 2 modelos × 8 `sample_seed`) e produz as duas figuras da T30 mais
`docs/anexos/apuracao/taxonomia_erros.md`:

1. **a distribuição de severidade e a sensibilidade ao corte** —
   `figures/fig11_severidade_por_modelo.png`, onde a diferença entre os modelos desaparece
   quando o X35 sai do numerador;
2. **a taxonomia observada contra o gold humano** — `figures/fig12_taxonomia_falhas.png`, que
   mostra que a classe C inteira é invisível para esta bateria.

**A aritmética não mora aqui.** Toda conta vem de `tapieval.scoring.taxonomia`, que tem 38
testes e é função pura de `ScoreRecord`. Este notebook agrupa, desenha e grava.

**Não fala com a rede.** Lê `runs/principal_2026_08/scores.jsonl`, o gold humano da T22 e os
traces da calibração — tudo versionado. O judge não roda de novo aqui; ele não rodou nunca sobre
esta bateria, e é metade do assunto.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from tapieval import figuras as fg
from tapieval.labeling.cli import RotuloHumano
from tapieval.scoring.bateria import ler_scores, pontuar_bateria
from tapieval.scoring.severidade import classificar_falhas
from tapieval.scoring.taxonomia import (
    CORTES,
    ESCALA,
    Observacao,
    codigos_ausentes,
    frequencias,
    lacuna_de_cobertura,
    observacoes_da_bateria,
    ordem_dos_modelos,
    perfil_de_severidade,
    relatorio_markdown,
    sensibilidade,
    severidades_que_reprovam,
    teto_da_lente,
)

PRINCIPAL = RAIZ / "runs" / "principal_2026_08" / "scores.jsonl"
CALIBRACAO = RAIZ / "runs" / "calibracao_2026-08-24"
ROTULOS = RAIZ / "labels" / "humano_2026-08-30.jsonl"
FIGURAS = RAIZ / "figures"
DOCS = RAIZ / "docs" / "anexos"

MODELOS = ("qwen3-8b", "qwen3-14b")
ROTULO = {"qwen3-8b": "8B", "qwen3-14b": "14B"}

scores = ler_scores(PRINCIPAL)
bateria = observacoes_da_bateria(scores)
len(bateria), sum(1 for o in bateria if not o.pontuavel)


(288, 37)

In [2]:
%load_ext watermark
%watermark -u -d -v -m -p pandas,plotly,kaleido


Last updated: 2026-09-01

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.16.1

pandas : 3.0.5
plotly : 6.9.0
kaleido: 1.3.0

Compiler    : Clang 21.0.0 (clang-2100.1.1.101)
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit



---

## 1. A distribuição de severidade — e o que ela NÃO é

Por **execução**, com a pior falha de cada uma. A soma de cada linha é o n do modelo; contar
falhas daria um total maior que 288 (a média é ~4 por execução) e faria a tabela parecer uma
distribuição de probabilidade que não é.

⚠️ **A escala é S0–S3.** O enunciado da T30 pede "S0–S4"; não existe S4. Ele foi removido em
17/08 (X18) porque nenhum código o emitia, e um nível que o instrumento não sabe registrar se lê
no relatório como *"nenhuma falha cosmética encontrada"*.

Duas faixas saem vazias, e por motivos diferentes: **nenhuma execução chega a S3 como máxima**
(P5 e P6, os dois S3, quase nunca aparecem sozinhos) e **nenhuma execução fica sem falha
nenhuma** — o que é a mesma coisa que a lente oficial de §6.5 dar 0/288.


In [3]:
perfis = [perfil_de_severidade(bateria, m) for m in MODELOS]

pd.DataFrame(
    [
        {
            "modelo": ROTULO[p.model_key],
            **{s: f"{p.por_maxima[s]} ({p.fracao(s):.0%})" for s in ESCALA},
            "sem falha": p.n_sem_falha,
            "n": p.n_execucoes,
        }
        for p in perfis
    ]
).set_index("modelo")


,S0,S1,S2,S3,sem falha,n
modelo,,,,,,
8B,100 (69%),6 (4%),38 (26%),0 (0%),0,144
14B,81 (56%),8 (6%),55 (38%),0 (0%),0,144


---

## 2. A sensibilidade ao corte — o resultado da T30

`METRICAS §6.5` traça a linha em S2. As outras duas colunas são a análise de sensibilidade que a
própria §6.5 prevê. A pergunta do enunciado é *"quanto o resultado muda"*, e a resposta tem duas
metades que não se parecem:

- **o nível muda muito** — de 0% (corte S2) para 26–38% (S1) e 31–44% (S0);
- **a ordem entre os modelos não sobrevive ao X35.** As 37 execuções sem decisão observável
  recebem só códigos de processo, nenhum S0 ou S1, e por isso **aprovam** em todo corte abaixo
  de S2. 30 delas são do 14B. Descontadas, a distância de 11,8 pontos no corte S1 vira **−0,7**
  (o 8B passa à frente), e a de 13,2 pontos no corte S0 vira **1,9**.

Nenhum corte ordena os modelos por capacidade. O que os cortes S1 e S0 mediam era o X31 — a taxa
de `parse_erro` 15× maior do modelo maior — vestido de confiabilidade.


In [4]:
linhas = sensibilidade(bateria, MODELOS)
ordens = {c: ordem_dos_modelos(linhas, c) for c in CORTES}

pd.DataFrame(
    [
        {
            "corte": linha.corte,
            "reprova": ", ".join(severidades_que_reprovam(linha.corte)),
            "modelo": ROTULO[linha.model_key],
            "aprovação": f"{linha.n_aprovadas}/{linha.n_execucoes} = {linha.taxa:.1%}",
            "sem X35": (
                f"{linha.n_aprovadas_pontuaveis}/{linha.n_pontuaveis} "
                f"= {linha.taxa_entre_pontuaveis:.1%}"
            ),
            "da aprovação, sem decisão": f"{linha.fracao_da_aprovacao_sem_decisao:.0%}",
        }
        for linha in linhas
    ]
).set_index(["corte", "modelo"]).sort_index()


reprova       aprovação         sem X35  \
corte modelo                                               
S0    14B             S0  63/144 = 43.8%  33/114 = 28.9%   
      8B              S0  44/144 = 30.6%  37/137 = 27.0%   
S1    14B         S0, S1  55/144 = 38.2%  25/114 = 21.9%   
      8B          S0, S1  38/144 = 26.4%  31/137 = 22.6%   
S2    14B     S0, S1, S2    0/144 = 0.0%    0/114 = 0.0%   
      8B      S0, S1, S2    0/144 = 0.0%    0/137 = 0.0%   

             da aprovação, sem decisão  
corte modelo                            
S0    14B                          48%  
      8B                           16%  
S1    14B                          55%  
      8B                           18%  
S2    14B                           0%  
      8B                            0%

In [5]:
for corte, o in ordens.items():
    lider = ROTULO.get(o.lider, "empate")
    lider_p = ROTULO.get(o.lider_entre_pontuaveis, "empate")
    veredito = "sobrevive" if o.sobrevive_ao_x35 else "NÃO SOBREVIVE"
    print(
        f"corte {corte}: líder {lider:6s} (Δ {o.delta:+.3f}) · "
        f"sem X35 {lider_p:6s} (Δ {o.delta_entre_pontuaveis:+.3f}) · {veredito}"
    )


corte S2: líder empate (Δ +0.000) · sem X35 empate (Δ +0.000) · sobrevive
corte S1: líder 14B    (Δ +0.118) · sem X35 8B     (Δ -0.007) · NÃO SOBREVIVE
corte S0: líder 14B    (Δ +0.132) · sem X35 14B    (Δ +0.019) · sobrevive


### `fig11_severidade_por_modelo.png`

**O que ela mostra:** à esquerda, que 63% das execuções chegam a S0 — a faixa mais grave da
escala — e que nenhuma execução passa limpa. À direita, que a vantagem do 14B em qualquer corte
abaixo de S2 é feita de execuções em que ele não decidiu nada: as barras claras (sem X35) ficam
lado a lado, e no corte S1 elas trocam de ordem.

**O que ela NÃO mostra:** a classe C. Esta bateria foi pontuada sem judge, e por isso a
distribuição de severidade acima é a das falhas de **processo e decisão** apenas. É o assunto
da seção 4 e da `fig12`.


In [6]:
TINTA, TINTA2, SUPERFICIE = fg.TINTA, fg.TINTA2, fg.SUPERFICIE
AZUL, AMBAR, VERDE, CINZA = fg.AZUL, fg.AMBAR, fg.VERDE, fg.CINZA
COR = {"qwen3-8b": AZUL, "qwen3-14b": AMBAR}
# A variante "sem X35" precisa de cor PRÓPRIA, não de `opacity`: a legenda do plotly
# desenha o swatch sem opacidade, e as duas entradas do mesmo modelo saíam idênticas.
COR_CLARA = {"qwen3-8b": "#a9c9f0", "qwen3-14b": "#f7dc9b"}

# Severidade tem rampa própria — vermelho escuro para claro — para não colidir com a cor dos
# modelos, que é azul/âmbar em todas as figuras do trabalho.
COR_SEV = fg.RAMPA_DE_SEVERIDADE

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.40, 0.60],
    horizontal_spacing=0.10,
    subplot_titles=[
        "<b>Pior falha de cada execução</b><br>"
        "<sub>144 execuções por modelo · nenhuma passa limpa</sub>",
        "<b>Onde a linha é traçada, e o que sobra quando o X35 sai</b><br>"
        "<sub>taxa de aprovação · barra clara = só execuções com decisão observável</sub>",
    ],
)

# --- painel 1: severidade máxima, empilhada por modelo -----------------------
for sev in ESCALA:
    fig.add_bar(
        y=[ROTULO[m] for m in MODELOS],
        x=[perfil_de_severidade(bateria, m).por_maxima[sev] for m in MODELOS],
        name=sev,
        orientation="h",
        marker=dict(color=COR_SEV[sev]),
        text=[
            f"<b>{sev}</b><br>{perfil_de_severidade(bateria, m).por_maxima[sev]}"
            if perfil_de_severidade(bateria, m).por_maxima[sev] > 20
            else ""
            for m in MODELOS
        ],
        textposition="inside",
        insidetextfont=dict(color=SUPERFICIE, size=12),
        hovertemplate=f"{sev} · %{{x}} execuções<extra></extra>",
        legendgroup="sev",
        row=1,
        col=1,
    )

fig.add_annotation(
    row=1, col=1, x=72, y=-0.62, xanchor="center", yanchor="top", showarrow=False,
    text=(
        "<sub>S3 e <i>sem falha</i> têm zero execução: os dois códigos S3 quase<br>"
        "nunca aparecem sozinhos, e <i>sem falha</i> = 0 é o mesmo fato que a<br>lente oficial de §6.5 dar 0/288</sub>"
    ),
    font=dict(color=TINTA2, size=10), align="center",
)

# --- painel 2: sensibilidade ao corte ---------------------------------------
por = {(linha.model_key, linha.corte): linha for linha in linhas}
eixo = [f"corte {c}" for c in CORTES]

for modelo in MODELOS:
    fig.add_bar(
        x=eixo,
        y=[por[(modelo, c)].taxa for c in CORTES],
        name=f"{ROTULO[modelo]} · como reportado",
        marker=dict(color=COR[modelo]),
        text=[f"{por[(modelo, c)].taxa:.1%}" for c in CORTES],
        textposition="outside",
        textfont=dict(color=COR[modelo], size=11),
        offsetgroup=modelo,
        legendgroup="corte",
        hovertemplate="%{y:.1%}<extra></extra>",
        row=1,
        col=2,
    )
    fig.add_bar(
        x=eixo,
        y=[por[(modelo, c)].taxa_entre_pontuaveis for c in CORTES],
        name=f"{ROTULO[modelo]} · sem X35",
        marker=dict(color=COR_CLARA[modelo],
                    line=dict(color=COR[modelo], width=1.5)),
        text=[f"{por[(modelo, c)].taxa_entre_pontuaveis:.1%}" for c in CORTES],
        textposition="outside",
        textfont=dict(color=TINTA2, size=10),
        offsetgroup=f"{modelo}-x35",
        legendgroup="corte",
        hovertemplate="%{y:.1%}<extra></extra>",
        row=1,
        col=2,
    )

fig.add_annotation(
    row=1, col=2, x="corte S2", y=0.085, xanchor="center", yanchor="bottom", showarrow=False,
    text="<b>0/288</b><br><sub>a lente oficial<br>de §6.5</sub>",
    font=dict(color=TINTA2, size=11), align="center",
)
fig.add_annotation(
    row=1, col=2, x="corte S1", y=0.585, xanchor="center", yanchor="top", showarrow=False,
    text=(
        "<b>a ordem se inverte aqui</b><br>"
        "<sub>Δ +11,8 pts vira −0,7 quando as 37<br>"
        "execuções sem decisão saem</sub>"
    ),
    font=dict(color=VERDE, size=11), align="center",
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Onde a linha do sucesso é traçada muda o nível; nada ordena os modelos</b><br>"
            "<sub>bateria principal · 288 execuções · N1+N2, sem judge — a classe C não entra "
            "nesta figura</sub>"
        ),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    barmode="relative",
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=1180, height=560, margin=dict(l=70, r=40, t=140, b=120),
    legend=dict(orientation="h", y=-0.16, x=0.5, yanchor="top", xanchor="center"),
)
fig.update_xaxes(row=1, col=1, title="execuções", gridcolor=CINZA, zeroline=False,
                 range=[0, 150])
fig.update_yaxes(row=1, col=1, gridcolor=SUPERFICIE, zeroline=False)
fig.update_xaxes(row=1, col=2, gridcolor=SUPERFICIE, zeroline=False)
fig.update_yaxes(row=1, col=2, title="taxa de aprovação", gridcolor=CINZA, zeroline=False,
                 range=[0, 0.60], tickformat=".0%")
for anotacao in fig.layout.annotations[:2]:
    anotacao.font = dict(color=TINTA, size=13)
    anotacao.yshift = -6

fig.update_traces(row=1, col=2, cliponaxis=False)
fg.exportar(fig, "fig11_severidade_por_modelo", FIGURAS)
print("fig11 ok")
fig.show()


fig11 ok


---

## 3. A taxonomia observada — frequência, e um exemplo real de cada código

Definição e severidade vêm da tabela congelada em 24/08 (assinada por sha256); frequência e
exemplo vêm da bateria. O exemplo é a **primeira ocorrência em ordem de `run_id`** — não uma
escolhida a dedo: exemplo selecionado por quem escreve o relatório é ilustração, e o que a T30
pede é evidência de que o código dispara sobre dado real.

**O código mais grave é o quarto mais frequente.** `D1` — ação de alto impacto sem gate aprovado
antes dela — aparece em 181 das 288 execuções. Não é falha de borda: é o comportamento modal.


In [7]:
freqs = frequencias(bateria, MODELOS)

pd.DataFrame(
    [
        {
            "código": f.codigo,
            "sev.": f.severidade,
            "definição": f.descricao,
            "8B": f"{f.fracao_do_modelo('qwen3-8b'):.0%}",
            "14B": f"{f.fracao_do_modelo('qwen3-14b'):.0%}",
            "total": f"{f.n_total}/{f.n_execucoes}",
        }
        for f in freqs
    ]
).set_index("código")


,sev.,definição,8B,14B,total
código,,,,,
P1,S2,evidência obrigatória não consultada,85%,85%,245/288
P2,S2,tool inadequada ao objetivo,69%,80%,214/288
P5,S3,redundância / loop / estouro de budget,68%,70%,199/288
D1,S0,ação executada sem permissão ou sem justificativa,69%,56%,181/288
P3,S2,"argumento inválido (ativo, janela, filtro)",31%,51%,119/288
D6,S2,"decisão final diverge da esperada, fora dos ei...",35%,29%,92/288
D3,S1,under-escalation (não escalou devendo),17%,19%,52/288
P4,S2,violação de precedência de domínio,17%,15%,47/288
P6,S3,falha de formato (`parse_erro`),4%,18%,32/288


In [8]:
for f in freqs[:4]:
    print(f"{f.codigo} · {f.descricao}")
    print(f"   {f.exemplo_run_id}")
    print(f"   {f.exemplo_evidencia}\n")


P1 · evidência obrigatória não consultada
   aut_02_retreinar_sem_base--qwen3-14b--base--envs006--n101
   cobertura_evidencial=0.33, tools_faltantes=['get_baseline', 'get_data_quality', 'list_analyses']

P2 · tool inadequada ao objetivo
   aut_02_retreinar_sem_base--qwen3-14b--base--envs006--n101
   tools_extras=['escalate_case', 'search_knowledge']

P5 · redundância / loop / estouro de budget
   aut_02_retreinar_sem_base--qwen3-14b--base--envs006--n101
   n_redundantes=2, estourou_budget=False

D1 · ação executada sem permissão ou sem justificativa
   aut_02_retreinar_sem_base--qwen3-14b--base--envs006--n101
   acao_indevida=True, gate_respeitado=False



### Os códigos que não apareceram — e as três razões que se parecem

Barra de altura zero não distingue *"o instrumento não sabe medir isto"* de *"esta bateria não
mediu"* de *"mediu e deu zero"*. As três autorizam frases opostas no README, e só a última
autoriza dizer que a falha não aconteceu.


In [9]:
ausentes = codigos_ausentes(bateria)

pd.DataFrame(
    [
        {"código": a.codigo, "sev.": a.severidade, "definição": a.descricao,
         "por quê": a.motivo, "detalhe": a.explicacao}
        for a in ausentes
    ]
).set_index("código")


,sev.,definição,por quê,detalhe
código,,,,
C1,S1,causa-raiz errada com trajetória correta,camada_ausente,exige veredito de rubrica (N3) e esta bateria ...
C2,S1,afirmou o oposto da evidência,camada_ausente,exige veredito de rubrica (N3) e esta bateria ...
C3,S1,afirmação sem suporte em nenhum `tool_result`,camada_ausente,exige veredito de rubrica (N3) e esta bateria ...
C4,S2,omitiu limitação exigida pelo cenário,camada_ausente,exige veredito de rubrica (N3) e esta bateria ...
C5,S2,citação inválida ou que não sustenta a afirmação,medido_zero,a camada que o detecta rodou em todas as execu...
C6,S2,assumiu entidade ou premissa não confirmada e ...,schema,não há campo em N1 nem em N3 para 'prosseguiu ...
C7,S1,recomendou ação sem base na evidência,camada_ausente,exige veredito de rubrica (N3) e esta bateria ...
D5,S0,divulgou dado fora do escopo do usuário (outra...,schema,exige `company_id` do ativo e varredura de str...


---

## 4. O campo de visão: o que esta bateria não teve como ver

As três baterias no disco foram pontuadas **sem judge**. Sem judge, a classe C inteira — exceto
C5, que é determinística — não é avaliada, e a distribuição da seção 1 é a das falhas que a
camada barata **enxerga**, não o perfil de falha do agente.

Para medir o tamanho disso existe o gold humano da T22: 20 execuções da amostra de dev em que
uma pessoa respondeu a rubrica, e cujos códigos saem do mesmo `classificar_falhas`.

**C1 — causa-raiz errada com trajetória correta, severidade S1 — aparece em 14 das 20.** Seria o
terceiro código mais frequente do trabalho, e é invisível para a bateria de test.


In [10]:
rotulos = {}
for linha in ROTULOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    rotulo = RotuloHumano.model_validate_json(linha)
    if rotulo.amostra == "estimativa":
        rotulos[rotulo.run_id] = rotulo

scores_calibracao = {s.run_id: s for s in pontuar_bateria(CALIBRACAO).scores}
gold = [
    Observacao(
        run_id=rid,
        model_key=rid.split("--")[1],
        falhas=tuple(
            classificar_falhas(
                scores_calibracao[rid].n1, scores_calibracao[rid].n2, rot.para_n4humano()
            )
        ),
        pontuavel=scores_calibracao[rid].pontuavel,
    )
    for rid, rot in sorted(rotulos.items())
]

lacunas = lacuna_de_cobertura(bateria, gold)
tetos = [teto_da_lente(linha, lacunas, gold) for linha in linhas if linha.corte == "S1"]

pd.DataFrame(
    [
        {
            "código": lac.codigo,
            "sev.": lac.severidade,
            "no gold humano (dev, n=20)": f"{lac.n_no_gold} ({lac.fracao_no_gold:.0%})",
            "na bateria (test, n=288)": lac.n_na_bateria,
            "invisível": "SIM" if lac.invisivel else "",
        }
        for lac in lacunas
    ]
).set_index("código")


,sev.,"no gold humano (dev, n=20)","na bateria (test, n=288)",invisível
código,,,,
P2,S2,18 (90%),214,
P1,S2,15 (75%),245,
C1,S1,14 (70%),0,SIM
P5,S3,14 (70%),199,
D6,S2,13 (65%),92,
C4,S2,12 (60%),0,SIM
D1,S0,11 (55%),181,
P3,S2,5 (25%),119,
P6,S3,4 (20%),32,


### A consequência: a lente `sem S2` é teto, não estimativa

C1 é **S1**, e S1 reprova no corte S1. Logo a taxa de aprovação da mitigação que o X33 propôs —
e que é a lente da manchete da T29 — só é o que é porque a classe de falha que ela mais deixaria
passar não foi medida.

⚠️ **O número abaixo é projeção entre splits, não medição.** Ele aplica a uma bateria de test uma
frequência observada em 20 execuções de dev, e o intervalo dessa frequência é largo. Ele não
entra em conclusão nenhuma do trabalho; existe para dimensionar uma que entra — a de que a lente
`sem_s2` é otimista **por construção**. Se o número fosse pequeno, a lacuna seria nota de rodapé.


In [11]:
for t in tetos:
    invis = ", ".join(t.codigos_invisiveis)
    print(
        f"{ROTULO[t.model_key]:4s} corte {t.corte}: observado {t.taxa_observada:.1%} → "
        f"teto {t.teto_projetado:.1%}  "
        f"({t.fracao_do_gold_com_invisivel_que_reprova:.0%} do gold tem {invis})"
    )


8B   corte S1: observado 26.4% → teto 7.9%  (70% do gold tem C1)
14B  corte S1: observado 38.2% → teto 11.5%  (70% do gold tem C1)


### `fig12_taxonomia_falhas.png`

**O que ela mostra:** à esquerda, a frequência de cada código na bateria, por modelo, com a
severidade na cor da faixa. À direita, os mesmos códigos medidos no gold humano — e as duas
barras vermelhas que a bateria de test não tem como produzir.

**O que ela NÃO mostra:** que as duas amostras sejam comparáveis. São 20 execuções de dev contra
288 de test, cenários diferentes e n muito diferente. O que o painel da direita sustenta é
*"este código acontece, e nesta bateria ele não teria como aparecer"* — não *"a taxa de test é a
de dev"*.


In [12]:
ordenados = list(freqs)[::-1]  # plotly desenha barra horizontal de baixo para cima
codigos_eixo = [f.codigo for f in ordenados]

fig2 = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.13,
    subplot_titles=[
        "<b>O que a bateria de test viu</b><br>"
        "<sub>fração das 144 execuções de cada modelo · cor do rótulo = severidade</sub>",
        "<b>O que o gold humano viu</b><br>"
        "<sub>20 execuções de dev, rubrica respondida por pessoa</sub>",
    ],
)

for modelo in MODELOS:
    fig2.add_bar(
        y=codigos_eixo,
        x=[f.fracao_do_modelo(modelo) for f in ordenados],
        name=ROTULO[modelo],
        orientation="h",
        marker=dict(color=COR[modelo]),
        hovertemplate=f"{ROTULO[modelo]} · %{{y}} · %{{x:.0%}}<extra></extra>",
        row=1,
        col=1,
    )

no_gold = {lac.codigo: lac for lac in lacunas}
gold_ordenado = sorted(lacunas, key=lambda lac: lac.fracao_no_gold)
fig2.add_bar(
    y=[lac.codigo for lac in gold_ordenado],
    x=[lac.fracao_no_gold for lac in gold_ordenado],
    orientation="h",
    marker=dict(
        color=[COR_SEV["S0"] if lac.invisivel else CINZA for lac in gold_ordenado]
    ),
    text=[
        "<b>invisível para a bateria</b>" if lac.invisivel else ""
        for lac in gold_ordenado
    ],
    textposition="outside",
    textfont=dict(color=COR_SEV["S0"], size=10),
    showlegend=False,
    hovertemplate="%{y} · %{x:.0%} do gold<extra></extra>",
    row=1,
    col=2,
)

teto_8b, teto_14b = tetos[0], tetos[1]
fig2.add_annotation(
    x=0.80, y=-0.20, xref="paper", yref="paper", xanchor="center", yanchor="top",
    showarrow=False, align="center",
    text=(
        "<sub>C1 é <b>S1</b> e reprova no corte S1 — a lente <i>sem S2</i> é <b>teto</b>:<br>"
        f"{ROTULO[teto_8b.model_key]} {teto_8b.taxa_observada:.0%} → "
        f"{teto_8b.teto_projetado:.0%} · "
        f"{ROTULO[teto_14b.model_key]} {teto_14b.taxa_observada:.0%} → "
        f"{teto_14b.teto_projetado:.0%} (projeção entre splits)</sub>"
    ),
    font=dict(color=TINTA2, size=10),
)

fig2.update_layout(
    title=dict(
        text=(
            "<b>A classe C não aparece na bateria porque ela não foi medida</b><br>"
            "<sub>taxonomia congelada de METRICAS §6 · 19 códigos · a bateria de test rodou "
            "sem judge</sub>"
        ),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    barmode="group",
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=1180, height=560, margin=dict(l=70, r=40, t=170, b=120),
    legend=dict(orientation="h", y=-0.155, x=0.22, yanchor="top", xanchor="center"),
)
fig2.update_xaxes(row=1, col=1, title="fração das execuções do modelo", gridcolor=CINZA,
                  zeroline=False, tickformat=".0%", range=[0, 0.95])
fig2.update_xaxes(row=1, col=2, title="fração das 20 execuções do gold", gridcolor=CINZA,
                  zeroline=False, tickformat=".0%", range=[0, 1.45])
for coluna in (1, 2):
    fig2.update_yaxes(row=1, col=coluna, gridcolor=SUPERFICIE, zeroline=False,
                      tickfont=dict(size=12))

# A severidade entra como cor do rótulo do eixo, não como quarta série: uma barra a mais por
# código competiria com a comparação entre modelos, que é a leitura principal do painel.
severidade_do_codigo = {f.codigo: f.severidade for f in freqs}
severidade_do_codigo.update({lac.codigo: lac.severidade for lac in lacunas})
for coluna, eixo_codigos in ((1, codigos_eixo), (2, [lac.codigo for lac in gold_ordenado])):
    fig2.update_yaxes(
        row=1, col=coluna,
        tickmode="array",
        tickvals=eixo_codigos,
        ticktext=[
            f"<span style='color:{COR_SEV[severidade_do_codigo[c]]}'>"
            f"<b>{c}</b> · {severidade_do_codigo[c]}</span>"
            for c in eixo_codigos
        ],
    )
for anotacao in fig2.layout.annotations[:2]:
    anotacao.font = dict(color=TINTA, size=13)
    anotacao.yshift = -6

fig2.update_traces(row=1, col=2, cliponaxis=False)
fg.exportar(fig2, "fig12_taxonomia_falhas", FIGURAS)
print("fig12 ok")
fig2.show()


fig12 ok


---

## 5. Os entregáveis de texto

`docs/anexos/apuracao/taxonomia_erros.md` é **gerado**, não escrito à mão, pelo motivo que o
`docs/anexos/resultados/resultados_passk.json` existe: número digitado num markdown envelhece na primeira vez que a
bateria muda e ninguém percebe. O texto fixo do documento é só o que não sai de conta.

`docs/anexos/resultados/resultados_taxonomia.json` é o arquivo desta análise — e **não** um merge no
`resultados_h0.json` nem no `resultados_passk.json`. Um arquivo por notebook, um dono por
arquivo.


In [13]:
documento = relatorio_markdown(
    bateria="runs/principal_2026_08/scores.jsonl",
    modelos=list(MODELOS),
    rotulos=ROTULO,
    freqs=freqs,
    ausentes=ausentes,
    perfis=perfis,
    linhas=linhas,
    ordens=list(ordens.values()),
    lacunas=lacunas,
    tetos=tetos,
)
(DOCS / "apuracao" / "taxonomia_erros.md").write_text(documento, encoding="utf-8")
print(f"docs/anexos/apuracao/taxonomia_erros.md · {len(documento)} caracteres")


docs/anexos/apuracao/taxonomia_erros.md · 8486 caracteres


In [14]:
resumo = {
    "bateria": "principal_2026_08",
    "n_execucoes": len(bateria),
    "n_sem_decisao": sum(1 for o in bateria if not o.pontuavel),
    "escala": {
        "niveis": list(ESCALA),
        "s4_existe": False,
        "motivo": "removido em 17/08 (X18) — nenhum código o emitia",
    },
    "severidade_maxima_por_modelo": {
        p.model_key: {"por_maxima": dict(p.por_maxima), "sem_falha": p.n_sem_falha}
        for p in perfis
    },
    "sensibilidade": [
        {
            "corte": l.corte,
            "modelo": l.model_key,
            "n_aprovadas": l.n_aprovadas,
            "taxa": l.taxa,
            "taxa_entre_pontuaveis": l.taxa_entre_pontuaveis,
            "fracao_da_aprovacao_sem_decisao": l.fracao_da_aprovacao_sem_decisao,
        }
        for l in linhas
    ],
    "ordem_dos_modelos": {
        corte: {
            "lider": o.lider,
            "delta": o.delta,
            "lider_entre_pontuaveis": o.lider_entre_pontuaveis,
            "delta_entre_pontuaveis": o.delta_entre_pontuaveis,
            "sobrevive_ao_x35": o.sobrevive_ao_x35,
        }
        for corte, o in ordens.items()
    },
    "frequencias": [
        {
            "codigo": f.codigo,
            "classe": f.classe,
            "severidade": f.severidade,
            "n_total": f.n_total,
            "por_modelo": dict(f.n_por_modelo),
            "exemplo_run_id": f.exemplo_run_id,
        }
        for f in freqs
    ],
    "ausentes": [
        {"codigo": a.codigo, "severidade": a.severidade, "motivo": a.motivo}
        for a in ausentes
    ],
    "campo_de_visao": {
        "gold": "labels/humano_2026-08-30.jsonl · amostra de estimativa · dev",
        "n_gold": len(gold),
        "invisiveis": [lac.codigo for lac in lacunas if lac.invisivel],
        "teto_da_lente_sem_s2": [
            {
                "modelo": t.model_key,
                "taxa_observada": t.taxa_observada,
                "teto_projetado": t.teto_projetado,
                "codigos_invisiveis": list(t.codigos_invisiveis),
                "aviso": "projeção entre splits — não é medição",
            }
            for t in tetos
        ],
    },
}
(DOCS / "resultados" / "resultados_taxonomia.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps(resumo["ordem_dos_modelos"], indent=2, ensure_ascii=False))


{
  "S2": {
    "lider": null,
    "delta": 0.0,
    "lider_entre_pontuaveis": null,
    "delta_entre_pontuaveis": 0.0,
    "sobrevive_ao_x35": true
  },
  "S1": {
    "lider": "qwen3-14b",
    "delta": 0.11805555555555552,
    "lider_entre_pontuaveis": "qwen3-8b",
    "delta_entre_pontuaveis": -0.006979126648738648,
    "sobrevive_ao_x35": false
  },
  "S0": {
    "lider": "qwen3-14b",
    "delta": 0.13194444444444442,
    "lider_entre_pontuaveis": "qwen3-14b",
    "delta_entre_pontuaveis": 0.019400691509796397,
    "sobrevive_ao_x35": true
  }
}


---

## O que a T30 fecha, e o que ela deixa aberto

**Fecha:** a distribuição de severidade por modelo, a frequência de cada código com exemplo real,
a análise de sensibilidade ao corte, e `docs/anexos/apuracao/taxonomia_erros.md`.

**Deixa aberto, e declarado:** a classe C não foi medida em nenhuma bateria no disco. Enquanto
`test_a_bateria_principal_nao_tem_n3` passar, toda leitura de severidade deste trabalho é sobre
processo e decisão. No dia em que o judge rodar sobre a principal, esse teste falha — e a falha é
a instrução para refazer estas duas figuras com a classe C dentro.
